In [1]:
import requests
import time
from bs4 import BeautifulSoup

import re

import pandas as pd
import urllib3
import os 

from datetime import date, timedelta

import traceback

from src.logger import logger


urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
outer_page_url = "https://dsebd.org/company_listing.php"
headers = {"User-Agent":"Mozilla/5.0"}

logger.info("pipeline started")

2026-08-21 14:58:23,574 | INFO | dse_pipeline | pipeline started


In [3]:
logger.info("Request send to outer URL")

response = requests.get(outer_page_url,headers=headers ,verify=False) 

if 200 <= response.status_code < 300:

    logger.info(
        "Request successful | status_code=%d | url=%s",
        response.status_code,
        outer_page_url
    )

else:

    logger.warning(
        "Request failed | status_code=%d | url=%s",
        response.status_code,
        outer_page_url
    )

2026-08-21 15:01:03,828 | INFO | dse_pipeline | Request send to outer URL
2026-08-21 15:01:06,804 | INFO | dse_pipeline | Request successful | status_code=200 | url=https://dsebd.org/company_listing.php


In [4]:
logger.info("Scraping the outer page URL")
soup = BeautifulSoup(response.text)

2026-08-21 15:03:28,705 | INFO | dse_pipeline | Scraping the outer page URL


In [5]:
#getting companies inner pages url
links = []

inner_base_url="https://dsebd.org/"

body = soup.find_all('div' , class_='BodyContent')

for section in body[:]:
    
    #filter out a tags only that contains href
    section_links=[link.get('href') for link in section.find_all('a', href=True)]
    
    links.extend(section_links)

In [6]:
links = list(map(lambda x: inner_base_url + x, links))

In [7]:
logger.info(f"Scraped {len(links)} companies from outer page URL")
len(links)

2026-08-21 15:03:32,862 | INFO | dse_pipeline | Scraped 636 companies from outer page URL


636

In [10]:
failed_requests=[]
processed_requests=[]

today = date.today()

def handle_request(link):

    try:
        response=requests.get(link, headers=headers, verify=False , timeout=30)
        # time.sleep(60)
        if 200 <= response.status_code < 300:
            processed_requests.append(link)
            return response.text
        else:
            failed_requests.append({
                    "link": link,
                    "status_code": response.status_code
                })
    
            logger.warning(
                "Request failed | status=%d | url=%s",
                response.status_code,
                link
                )
    
            return None
    except requests.exceptions.Timeout:
        failed_requests.append({
            "link": link,
            "status_code": "TIMEOUT"
        })

        logger.warning(
            "Request timeout  | url=%s",
            link
        )
        return None
        

def download_html_page(response, file_name):
    
    raw_folder = f"raw_data/{today}"
    os.makedirs(raw_folder, exist_ok=True)
    
    with open(f'{raw_folder}/{file_name}.html','w', encoding='utf-8') as f:
        f.write(response)
    
    
get_company_name=lambda x: x.split('=')[1]


In [11]:
logger.info("Sending download process | total_urls =%d",
           len(links)
           )

for i, inner_link in enumerate(links):
    
    if inner_link in processed_requests:
        print("Already Downloaded: ",inner_link)
        continue

    print("Downloading : ",inner_link)

    file_name=get_company_name(inner_link)
    
    response=handle_request(inner_link)
    
    if response is not None:

        download_html_page(
            response,
            file_name
        )
    if i % 50 == 0:

        logger.info(
            "Download progress | processed=%d/%d | successful=%d | failed=%d",
            i,
            len(links),
            len(processed_requests),
            len(failed_requests)
        )

logger.info(
    "Download process completed | total=%d | successful=%d | failed=%d",
    len(links),
    len(processed_requests),
    len(failed_requests)
)   

2026-08-21 15:20:40,729 | INFO | dse_pipeline | Sending download process | total_urls =636


2026-08-21 15:20:42,065 | INFO | dse_pipeline | Download progress | processed=0/636 | successful=1 | failed=0


2026-08-21 15:22:15,433 | INFO | dse_pipeline | Download progress | processed=50/636 | successful=51 | failed=0


2026-08-21 15:23:36,828 | INFO | dse_pipeline | Download progress | processed=100/636 | successful=101 | failed=0


2026-08-21 15:25:05,616 | INFO | dse_pipeline | Download progress | processed=150/636 | successful=151 | failed=0


...

2026-08-21 15:32:03,380 | INFO | dse_pipeline | Download progress | processed=400/636 | successful=401 | failed=0


2026-08-21 15:32:50,475 | INFO | dse_pipeline | Download progress | processed=450/636 | successful=451 | failed=0


2026-08-21 15:33:42,110 | INFO | dse_pipeline | Download progress | processed=500/636 | successful=501 | failed=0


2026-08-21 15:34:35,793 | INFO | dse_pipeline | Download progress | processed=550/636 | successful=551 | failed=0


2026-08-21 15:35:27,219 | INFO | dse_pipeline | Download progress | processed=600/636 | successful=601 | failed=0


In [12]:
len(processed_requests)

2026-08-21 15:36:34,834 | INFO | dse_pipeline | Download process completed | total=636 | successful=636 | failed=0


636

In [14]:
if len(os.listdir(f'raw_data/{today}')) == len(processed_requests):
    logger.info('Processed requests equal to files downloaded')
else:
    logger.warning("Processed Requests not equal to downloaded files")

2026-08-21 15:50:07,951 | INFO | dse_pipeline | Processed requests equal to files downloaded


###INNER PAGE CODE

In [16]:
table_6_quarter_types=['Q1','Q2','Half Yearly', 'Q3', '9 Months', 'Annual']
table_6_columns=[
 ("EPS","Basic"),
 ("EPS", "Diluted"),
 ("EPS - continuing operations","Basic"),
 ("EPS - continuing operations","Diluted"),
 ("EPS - continuing operations","Market price per share at period end"),
]

table_10_columns = [
    ("(P/E) ratio - EPS", "Basic Original"),
    ("(P/E) ratio - EPS", "Basic Restated"),
    ("(P/E) ratio - EPS", "Diluted"),
    ("(P/E) ratio - EPS Continuing Operations", "Basic Original"),
    ("(P/E) ratio - EPS Continuing Operations", "Basic Restated"),
    ("(P/E) ratio - EPS Continuing Operations", "Diluted"),
    ("Dividend", "-"),
    ("Dividend Yield", "-")
]

table_9_columns = [
    ("EPS", "Basic Original"),
    ("EPS", "Basic Restated"),
    ("EPS", "Diluted"),
    ("EPS Continuing Operations", "Basic Original"),
    ("EPS Continuing Operations", "Basic Restated"),
    ("EPS Continuing Operations", "Diluted"),
    ("NAV Per Share","Original"),
    ("NAV Per Share","Restated"),
    ("NAV Per Share","Diluted"),
    ("Profit/Loss and OCI", "PCO"),
    ("Profit/Loss and OCI", "Profit for the year (mn)"),
    ("Profit/Loss and OCI", "TCI")
]

FIELD_MAP = {

    "Scrip Code" : "Script Code",
    "Website": "website",
    "Web Address": "website",

    "Telephone No.": "telephone",
    "Contact Phone": "telephone",

    "E-mail": "email",

    "Address": "head_office_address",
    "Factory":"factory_address",

    "Address_Head Office": "head_office_address",
    "Address_Factory": "factory_address",

    "Concern Department": "department",

    "Fax": "fax",
    "Cell No.": "cell_no",
    "Trading Code" :"trading code",
    "Securities Trading Code": "trading code"
}

In [17]:
today = date.today()
def read_file(link):
    with open(f'raw_data/{today}/{link}', 'r',encoding="utf-8") as f:
        response=f.read()
    return response


def get_heading(tables_div, data):
    heading = tables_div[0].find("h2")

    if heading is None:
        return data

    text = heading.get_text(" ", strip=True)

    if ":" in text:
        key, value = text.split(":", 1)
    else:
        key = text
        value = ""

    key = key.strip()
    value = value.strip()

    data["type"] = key          # "Company Name" or "Security Name"
    data["name"] = value        # Actual name

    return data
    

def get_codes_table_0(table,data):
    for key_value in table.find_all(['th','td']):
        text = key_value.get_text(strip=True)

        if not text:          # Skip empty cells
            continue

        if ":" not in text:   # Skip cells without key:value
            continue

        key, value = text.split(":", 1)

        normalized_key = FIELD_MAP.get(key, key)
        # data[normalized_key] = value

        data[normalized_key.strip()] = value.strip()

        # print(normalized_key.strip() , " : ", value.strip())
    return data
   

def handle_market_basic_info(table, data):
    rowspan_key = None

    for row in table.find_all('tr', recursive=False):

        cells = row.find_all(['th', 'td'], recursive=False)

        if row.find("select"):
            continue

        if len(cells) < 2:
            continue

        # If previous row had rowspan, insert that key back
        if len(cells) % 2 != 0 and rowspan_key is not None:
            cells.insert(0, rowspan_key)

        for i in range(0, len(cells), 2):

            # Remember current rowspan key
            if cells[i].name == "th" and cells[i].has_attr("rowspan"):
                rowspan_key = cells[i]

            key = cells[i].get_text(strip=True)

            if cells[i + 1].find('table'):
                nested_table = cells[i + 1].find('table')
                values = [td.get_text(strip=True) for td in nested_table.find_all('td')]

                data[key] = {
                    "value": values[0],
                    "percentage": values[1]
                }

            else:
                value = cells[i + 1].get_text(strip=True)
                data[key] = value

    return data


def handle_agm_info(table, data,company):
    data_nested={}
    for row in table.find_all('tr', recursive=False):
        cells = row.find_all(['th','td'], recursive=False)
        
        if len(cells) < 2:
            continue
        
        
        for i in range(0, len(cells), 2):
            key = cells[i].get_text(strip=True)
            
            value = cells[i+1].get_text(strip=True)
            # if "%" in value:
            if key in ["Cash Dividend", "Bonus Issue (Stock Dividend)"]:
                data_nested[key]=[]
                # print(repr(key))
    
                separated_data = value.split(",")
                for item in separated_data:
                    parts = item.strip().split()
                    if len(parts) != 2:
                        continue
                    value1, value2 = parts
                    
                    if re.fullmatch(r'\d+',value1):
                        obj={
                            "year": value2, 
                            "percentage":value1,
                            "trading code":company
                        }
                        data_nested[key].append(obj)
                        # print("Header :",key, "percentage :", value1, "year :",value2)
                        
                        
                    else:
                        obj={
                           "year": value2, 
                            "percentage":value1,
                            "trading code":company
                        }
                        data_nested[key].append(obj)
                        # print("Header :",key, "percentage :", value1, "year :",value2)
                       
                                
            else:
                value = cells[i+1].get_text(strip=True)
                data[key] = value
    return data_nested.get('Cash Dividend'), data_nested.get('Bonus Issue (Stock Dividend)') , data


def handle_interim_financial_performance(table, data,company):
    i=0
    for row in table.find_all('tr')[4:]:
        cells=row.find_all(['th','td'])
        if len(cells) == 1:
            continue
        else:
            values=[value.get_text(strip=True) for value in cells[1:]]
            # print(i, " : ",values)
            # print(i)
            quarter_values=list(zip(table_6_quarter_types, values))
            # print(columns[i], quarter_values)
            for quarter_value in quarter_values:
                obj={
                    "trading code":company,
                    "section":table_6_columns[i][0],
                    "metric":table_6_columns[i][1],
                    "quarter_type":quarter_value[0],
                    "value":quarter_value[1]
                }
    
                data.append(obj)
    
            i=i+1
    return data

                           

def handle_price_earnings(table,data,company):
    statement_type=None
    
    child_table=table
    
    parent_div=child_table.find_parent('div')
    h2_sibling = parent_div.find_previous_sibling('h2').get_text(strip=True)
    # print(h2_sibling.lower())
    
    if "un-audited" in h2_sibling.lower():
        statement="un-audited"
    else:
        statement="audited"

    tabular_data=[]
    for row in table.find_all('tr'):
        tds = [td.get_text(strip=True) for td in row.find_all('td')]
        tabular_data.append(tds)


    sub_headers= tabular_data[0][1:]

    
    for row in tabular_data[1:]:
        
        header= row[0]
    
        for year, value in zip(sub_headers, row[1:]):
            
            obj = {
            "trading code": company,
            "statement_type": statement,
            "section": header,
            "date": year,
            "value": value
            }
            data.append(obj)


In [18]:


def handle_financial_performance(data, table, columns, starting_row ,company):
    for row in table.find_all('tr')[starting_row:]:
        # print(row)
        tds=[td.get_text(strip=True) for td in row.find_all('td')]
        year=tds[0]
        values=tds[1:]
        
        for (section , metric), value in zip(columns, values):
            obj={
             'trading code':company,
              'year':year,
              'section': section,
               'metric':metric,
              'value':value
            }
            data.append(obj)
    return data




def handle_listing_info(table,data , history,company):
    for row in table.find_all('tr', recursive=False):
        cells = row.find_all(['td','th'])
    
        #if there are two cells then normal key value pair
        if len(cells) == 2:
            # print(cells[0].get_text(strip=True) ," : ",cells[1].get_text(strip=True))
            key= cells[0].get_text(strip=True)
            value= cells[1].get_text(strip=True)
            data[key]=value
    
        elif row.find('table'):
            obj={}
            header = " ".join(cells[0].get_text().split())
            match= re.search(r'as on\s+([A-Za-z]{3}\s+\d{1,2},\s+\d{4})', header)
            
            date = match.group(1) if match else None
            
            nested_table=row.find('table')
            keys_values=[key_value.get_text(strip=True).strip().split(':') for key_value in nested_table.find_all(['th','td'])]
          
            for key_value in keys_values:
                obj = {
                "trading code": company,
                "date": date,
                "holder_type": key_value[0],
                "share": key_value[1]
                }

                history.append(obj)
    return data
    


    
def handle_company_data(table, company_data , secretary_data):

    is_secretary = False

    for row in table.find_all("tr", recursive=False):
        cells = row.find_all("td", recursive=False)


        if len(cells) == 3:
            key = cells[0].get_text(strip=True)
            subkey = cells[1].get_text(strip=True)
            value = cells[2].get_text(" ", strip=True)
            full_key= key + "_" + subkey

            # print("Before normalization :", full_key)

            normalized_key = FIELD_MAP.get(full_key, key)

            company_data[f"{normalized_key}"] = value
            # print(f"{normalized_key}", value)
            
        elif len(cells) >= 2:
            key = cells[0].get_text(strip=True)
            value = cells[1].get_text(" ", strip=True)
        
            if key == "Company Secretary Name":
                is_secretary = True
        
            if is_secretary:
                # print("secretary presnt ")
                secretary_data["trading code"]=company_data["trading code"]
                secretary_data[key] = value
            else:
                normalized_key = FIELD_MAP.get(key, key)
                company_data[normalized_key] = value
                # print(normalized_key, value)
                # company_data[key] = value

        else:
            print("Skipping row:", [c.get_text(" ", strip=True) for c in cells])

    return company_data,secretary_data


def handle_corporate_status(table, data, company):
    section = None
    date = None

    for row in table.find_all("tr", recursive=False):
        cells = row.find_all(["td", "th"], recursive=False)

        if not cells:
            continue

        # Section header
        if len(cells) == 1:
            section = cells[0].get_text(" ", strip=True)

            match = re.search(r"as on (.+)", section)
            if match:
                date = match.group(1)
                section = "Present Loan Status"
            else:
                date = None

        # Normal key-value row
        elif len(cells) == 2:
            metric = cells[0].get_text(" ", strip=True)
            value = cells[1].get_text(" ", strip=True)

            data.append({
                "trading code": company,
                "section": section,
                "date": date,
                "metric": metric,
                "value": value
            })

        # Row with blank first column (rowspan)
        elif len(cells) == 3:
            metric = cells[1].get_text(" ", strip=True)
            value = cells[2].get_text(" ", strip=True)

            data.append({
                "trading code": company,
                "section": section,
                "date": date,
                "metric": metric,
                "value": value
            })

    # return data


In [19]:
def get_table_heading(table):
    #if nested table then return '' 
    if table.find_parent("table"):
        return '' 
         
    parent= table.find_parent('div')
    heading = parent.find_previous_sibling('h2')

    if heading is None:
        return ""
    return heading.get_text(strip=True)

In [20]:
processed_requests=[]

In [21]:
len(processed_requests)

0

In [23]:
logger.info("Extracting data from inner pages URL")
data={
    'companies':[],
    'market_info':[],
    'basic_info':[],
    'agm_info':[],
    'cash_dividend':[],
    'bonus_issues':[],
    'price_earnings':[], 
    'listing_info':[],
    'shareholder_history':[],
    'company_secretaries': [],
    'corporate_status':[],
    'financial_performance':[],
    'interim_financial_performance':[]
    
}

table_17_len = set()
skipping=0
i=1
for idx,inner_link in enumerate(os.listdir(f'raw_data/{today}')):

    try:

        if inner_link.startswith('.ipynb'):
            continue

        if inner_link in processed_requests:
            print(f'Data already processed : {inner_link}')
            continue
        
        company={}
        market_info={}
        basic_info={}
        agm_info={}
        cash_dividend=[]
        bonus_issues={}
        listing_info={}
        company_secretary={}
        # corporate_status=[]
        financial_performance=[]
        interim_financial_performance=[]
    
        # print(inner_link)
    
        inner_page_response=read_file(inner_link)
        
        
        # inner_page_reponse = handle_request(inner_link)
        inner_page_soup = BeautifulSoup(inner_page_response)
    
        # get elements
        outer_div= inner_page_soup.find_all('div', class_='gray')
    
        if not outer_div:
            print(f"Skipping {inner_link}: RightBody container not found")
            skipping =skipping + 1
            continue
    
        main_div = outer_div[0].find('div', id='RightBody')
    
        if main_div is None:
            print(f"Skipping {inner_link}: RightBody div missing")
            skipping =skipping + 1
            continue
    
        # main_div= outer_div[0].find('div', attrs={'id':'RightBody'})
        
        tables_div = main_div.find_all('div', class_='row')
        tables = tables_div[0].find_all('table') #cant do recursive=False output ->0
    
        # print("Length of tables ",len(tables))
    
    
        #get heading info
        company = get_heading(tables_div,company)
        print(idx , " : ", inner_link , " len :  " ,len(tables))

    
        if company["type"]== "Security Name":
            company = get_codes_table_0(tables[0],company)

            for table in tables[1:]:
                heading  = get_table_heading(table).strip()
                print(heading)

                if 'Market Information' in heading :
                    market_info = handle_market_basic_info(table, market_info)
                    market_info["trading code"]=company['trading code']
                    # data["market_info"].append(market_info)

                    
                elif 'Basic Information' in heading:
                    basic_info = handle_market_basic_info(table, basic_info)
                    basic_info["trading code"]=company['trading code']
                    # data["basic_info"].append(basic_info)

                elif 'Address' in heading:
                    company, company_secretary = handle_company_data(
                    table, company, company_secretary )

                    if company_secretary:
                        data["company_secretaries"].append(company_secretary)
                    
                else:
                    continue
            
            data["companies"].append(company)

            data["market_info"].append(market_info)

            data["basic_info"].append(basic_info)


            
        else:
    
            
            #get table 0 data
            company = get_codes_table_0(tables[0],company)

            #per page table loop
            for idx, table in enumerate(tables):

                if idx == 11:
                    #handle table 11
                    company =  handle_market_basic_info(tables[11], company)
                    continue

                heading =  get_table_heading(table).strip()

                print(heading)

                
                #error
                if 'Market Information' in heading :
                    #handle table 1 
                    market_info = handle_market_basic_info(table, market_info)
                    print(market_info , company['trading code'])
                    market_info["trading code"]=company['trading code']

                elif 'Basic Information' in heading:
                    basic_info = handle_market_basic_info(table, basic_info)
                    basic_info['trading code']=company['trading code']

                elif 'AGM' in heading:
                    cash_dividend , bonus_issues , agm_info = handle_agm_info(table,agm_info,company['trading code'])
                    agm_info['trading code']=company['trading code']

                elif 'Interim Financial Performance' in heading:
                    interim_financial_performance = handle_interim_financial_performance(table , interim_financial_performance, company['trading code'])

                #handles two tables
                elif 'Price Earnings (P/E)' in heading:
                    handle_price_earnings(table, data['price_earnings'], company['trading code'])

                elif 'Financial Performance as per Audited Financial Statements as per IFRS/IAS or BFRS/BAS' == heading:
                    handle_financial_performance(financial_performance, table ,table_9_columns,3, company['trading code'])

                elif 'Financial Performance... (Continued)' == heading:
                    handle_financial_performance(financial_performance,table ,table_10_columns,4,company['trading code'])

                elif 'Other Information of the Company' == heading:
                    listing_info = handle_listing_info(table, listing_info, data['shareholder_history'],company['trading code'])
                    listing_info['trading code']=company['trading code']

                elif 'Corporate Performance' in heading:
                    handle_corporate_status(table ,data["corporate_status"], company['trading code'])

                elif 'Address' in heading:
                    company, company_secretary = handle_company_data( table, company, company_secretary )

                else:
                    continue


            print("_____________________________________________")
        
    
            data["companies"].append(company)
            data["market_info"].append(market_info)
            data["basic_info"].append(basic_info)
            data["cash_dividend"].extend(cash_dividend or [])
            data["bonus_issues"].extend(bonus_issues or [])
            data["agm_info"].append(agm_info)
            data["listing_info"].append(listing_info)
            if company_secretary:
                data["company_secretaries"].append(company_secretary)
            # data["corporate_status"].append(corporate_status)
            data["financial_performance"].extend(financial_performance or [])
            data["interim_financial_performance"].extend(interim_financial_performance or [])
    
        processed_requests.append(inner_link)
        
        
        # print(idx, "-:",inner_link)
        if idx % 50 == 0:

            logger.info(
                "Extraction progress | processed=%d/%d | successful=%d ",
                idx,
                len(links),
                len(processed_requests)
            )
        print("___________________________________________")
        
    except Exception as e:
        print(f"ERROR in {inner_link}: {e}")
        logger.warning("Error in extraction : %e", e)
        traceback.print_exc()
    
logger.info("Extraction Completed")

2026-08-21 16:01:39,183 | INFO | dse_pipeline | Extracting data from inner pages URL


0  :  1JANATAMF.html  len :   19
Company Name:First Janata Bank Mutual Fund
Market Information:Aug 20, 2026
{'Last Trading Price': '3.60', "Day's Range": '3.60 - 3.70', 'Last Update': '2:40 PM', "52 Weeks' Moving Range": '2.10 - 4.30', 'Change*': {'value': '0', 'percentage': '0.00%'}, "Day's Trade (Nos.)": '107', "Yesterday's Closing Price": '3.60', "Day's Volume (Nos.)": '743,851', 'Adjusted Opening Price': '3.60', "Day's Value (mn)": '2.69', 'Opening Price': '3.70', 'Market Capitalization (mn)': '1,043.724', 'Closing Price': '3.60', 'Free Float Market Cap. (mn)': '0.000'} 1JANATAMF

Basic Information

Last AGM held on:For the year ended:

                Jun 30, 2024
Interim Financial Performance:2025
Price Earnings (P/E) Ratio based on latest Un-audited Financial Statements
Price Earnings (P/E) Ratio Based on latest Audited Financial Statements
Financial Performance as per Audited Financial Statements as per IFRS/IAS or BFRS/BAS
Financial Performance... (Continued)
Other Information

2026-08-21 16:05:55,135 | INFO | dse_pipeline | Extraction progress | processed=400/636 | successful=401 


400  :  TB10Y0531.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomalies are found , please inform us (phone: +880241040189-200, +8809666702070, Ext- 1546).']
___________________________________________
401  :  TB10Y0532.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomal

2026-08-21 16:06:21,925 | INFO | dse_pipeline | Extraction progress | processed=450/636 | successful=451 


...

500  :  TB20Y0434.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomalies are found , please inform us (phone: +880241040189-200, +8809666702070, Ext- 1546).']
___________________________________________
501  :  TB20Y0436.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomal

2026-08-21 16:07:18,533 | INFO | dse_pipeline | Extraction progress | processed=550/636 | successful=551 


550  :  TB20Y1029.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomalies are found , please inform us (phone: +880241040189-200, +8809666702070, Ext- 1546).']
___________________________________________
551  :  TB20Y1030.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomal

2026-08-21 16:07:47,792 | INFO | dse_pipeline | Extraction progress | processed=600/636 | successful=601 


600  :  TB5Y1126.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomalies are found , please inform us (phone: +880241040189-200, +8809666702070, Ext- 1546).']
___________________________________________
601  :  TB5Y1127.html  len :   6
Market Information:Aug 20, 2026

Basic Information
Address of the Issuer
Address of the Issuer
Skipping row: ['Note: (i) All dynamic information of this page are only valid for Public and Debt board. (ii) Annual/ Audited figures are updated after AGM except for ‘Total No. of Outstanding Securities’ which is updated on record date. Disclaimer: This page is now under updating process. If any anomalie

SAVING PROCESSED DATA

In [26]:
#saving processed data
logger.info("Extraction Completed")
today = date.today()
output_folder =f'processed_data/{today}'
os.makedirs(output_folder, exist_ok=True)


for column in data.keys():
    path =os.path.join(output_folder, f'{column}.csv')
    print(path)
    df=pd.DataFrame(data[column])
    df.to_csv(path, index=False)
    logger.info("Saved file : %s", column)
    


2026-08-21 16:19:18,222 | INFO | dse_pipeline | Extraction Completed
2026-08-21 16:19:18,242 | INFO | dse_pipeline | Saved file : companies
2026-08-21 16:19:18,264 | INFO | dse_pipeline | Saved file : market_info
2026-08-21 16:19:18,283 | INFO | dse_pipeline | Saved file : basic_info
2026-08-21 16:19:18,292 | INFO | dse_pipeline | Saved file : agm_info
2026-08-21 16:19:18,304 | INFO | dse_pipeline | Saved file : cash_dividend
2026-08-21 16:19:18,317 | INFO | dse_pipeline | Saved file : bonus_issues
2026-08-21 16:19:18,407 | INFO | dse_pipeline | Saved file : price_earnings


processed_data/2026-08-21\companies.csv
processed_data/2026-08-21\market_info.csv
processed_data/2026-08-21\basic_info.csv
processed_data/2026-08-21\agm_info.csv
processed_data/2026-08-21\cash_dividend.csv
processed_data/2026-08-21\bonus_issues.csv
processed_data/2026-08-21\price_earnings.csv


2026-08-21 16:19:18,414 | INFO | dse_pipeline | Saved file : listing_info
2026-08-21 16:19:18,446 | INFO | dse_pipeline | Saved file : shareholder_history
2026-08-21 16:19:18,456 | INFO | dse_pipeline | Saved file : company_secretaries
2026-08-21 16:19:18,484 | INFO | dse_pipeline | Saved file : corporate_status


processed_data/2026-08-21\listing_info.csv
processed_data/2026-08-21\shareholder_history.csv
processed_data/2026-08-21\company_secretaries.csv
processed_data/2026-08-21\corporate_status.csv
processed_data/2026-08-21\financial_performance.csv


2026-08-21 16:19:18,699 | INFO | dse_pipeline | Saved file : financial_performance
2026-08-21 16:19:18,790 | INFO | dse_pipeline | Saved file : interim_financial_performance


processed_data/2026-08-21\interim_financial_performance.csv


CAPTURE DATA CHANGE

In [269]:
def log_changes(old_df , new_df , changes):
    for trading_code in new_df.index:
        if trading_code not in old_df.index:
            print("New company added ->Trading Code : ",trading_code)
            print("________________________________________________________________")
            continue
    
        for column in new_df.columns:
            old_value = old_df.loc[trading_code, column]
            new_value = new_df.loc[trading_code, column]
    
            if pd.isna(old_value):
                old_value = ''
    
            if pd.isna(new_value):
                new_value = ''
    
            if old_value != new_value:
                changes.append({
                    "trading code" : trading_code,
                    "column" : column, 
                    "old_value" :old_value ,
                    "new_value" : new_value
                })
    return changes
                

In [278]:
from pathlib import Path

APP_DIR = Path("/app")

yesterday = today - timedelta(days = 1 )

old_folder_path = APP_DIR / "processed_data" / str(yesterday)
new_folder_path = APP_DIR / "processed_data" / str(today)



# old_folder_path = rf'processed_data\{yesterday}'
# new_folder_path = rf'processed_data\{today}'

#iterate through all files and set index for each to compare to other days file
changes= []

files= ['agm_info.csv', 'companies.csv' , 'company_secretaries.csv' , 'basic_info.csv' , 'listing_info.csv' , 'market_info.csv',
       'cash_dividend.csv' , 'bonus_issues.csv' ,'shareholder_history.csv', 'price_earnings.csv', 'interim_financial_performance.csv',
       'financial_performance.csv', 'corporate_status.csv']
if not old_folder_path.exists():
    print(f"No previous dataset found: {old_folder_path}")
    print("Skipping change detection.")
    print("Today's data will be used as the baseline.")

else:
    for file in files:
        old_file_path = old_folder_path+ '\\' + file
        new_file_path = new_folder_path+ '\\' + file
        
        old_df = pd.read_csv(old_file_path)
        new_df = pd.read_csv(new_file_path)
    
    
        print("Checking changes ->", file)
    
        if file in [
        'agm_info.csv',
        'companies.csv',
        'company_secretaries.csv',
        'basic_info.csv',
        'listing_info.csv',
        'market_info.csv'
        ]:
            old_df = old_df.set_index("trading code")
            new_df = new_df.set_index("trading code")
    
    
        elif file == 'cash_dividend.csv' or file == 'bonus_issues.csv':
            old_df["year"] = old_df["year"].astype('int')
            new_df["year"] = new_df["year"].astype('int')
    
            old_df = old_df.set_index(["trading code","year" ])
            new_df = new_df.set_index(["trading code","year" ])
    
        elif file == 'shareholder_history.csv':
            old_df = old_df.set_index(['trading code','date' , 'holder_type' ])
            new_df = new_df.set_index(['trading code','date' , 'holder_type'  ])
    
        elif file == 'price_earnings.csv':
            old_df = old_df.set_index(['trading code', 'statement_type' , 'section','date' ])
            new_df = new_df.set_index(['trading code', 'statement_type' , 'section','date' ])
    
        elif file == 'interim_financial_performance.csv':
            old_df = old_df.set_index(['trading code', 'section', 'metric' , 'quarter_type' ])
            new_df = new_df.set_index(['trading code', 'section', 'metric' , 'quarter_type' ])
    
        elif file == 'financial_performance.csv':
            old_df = old_df.set_index(['trading code', 'section', 'metric' , 'year' ])
            new_df = new_df.set_index(['trading code', 'section', 'metric' , 'year' ])
    
        elif file == 'corporate_status.csv':
            old_df = old_df.fillna('-')
            old_df = old_df.set_index(['trading code', 'section', 'metric' , 'date'])
    
            new_df =  new_df.fillna('-')
            new_df = new_df.set_index(['trading code', 'section', 'metric' , 'date'])
            
        log_changes(old_df , new_df , changes)



2026-07-21 2026-07-20
Checking changes -> agm_info.csv
Checking changes -> companies.csv
Checking changes -> company_secretaries.csv
Checking changes -> basic_info.csv
Checking changes -> listing_info.csv
Checking changes -> market_info.csv
Checking changes -> cash_dividend.csv
Checking changes -> bonus_issues.csv
Checking changes -> shareholder_history.csv
Checking changes -> price_earnings.csv
Checking changes -> interim_financial_performance.csv
Checking changes -> financial_performance.csv
Checking changes -> corporate_status.csv


In [281]:
#save changes to a folder
if changes :
    df_changes = pd.DataFrame(changes)

    changes_folder=f'changes/{today}'
    os.makedirs(changes_folder, exist_ok=True)

    # will store the changed data of today and yesterdays file
    file_path = os.path.join(
        changes_folder,
        'changes.xlsx'
    )

    df_changes.to_excel(
        file_path,
        index=False
    )


else:
    print('No changes detected')
    
    

No changes detected


In [8]:
from datetime import date

AttributeError: type object 'datetime.date' has no attribute 'now'

In [15]:
print(os.listdir("src"))

['.ipynb_checkpoints', 'logger.ipynb', 'logs']
